# コンピュータがどのように文字を扱っているか

コンピュータでは全てのデータを例外なく数値的に扱う。さあ、世界中の文字をどう扱おうか。

## 文字コード

文字一つ一つへのIDの割り当て方。

「文字列を数値化する方法」なんて、文字一個一個にID（数値）割り振ってそれを並べればいいやん、って、思う。実際、基本的にはその方法でいい。文字コードはその割り当て方のことで、用途によって幾つかのバリエーションがある。文字に割り当てられたIDは**コードポイント**と呼ぶ。

### ASCII

アルファベット、数字、その他いくつかの記号128文字を含んだ最小限の文字コード。

In [1]:
!ascii -d

    0 NUL    16 DLE    32      48 0    64 @    80 P    96 `   112 p 
    1 SOH    17 DC1    33 !    49 1    65 A    81 Q    97 a   113 q 
    2 STX    18 DC2    34 "    50 2    66 B    82 R    98 b   114 r 
    3 ETX    19 DC3    35 #    51 3    67 C    83 S    99 c   115 s 
    4 EOT    20 DC4    36 $    52 4    68 D    84 T   100 d   116 t 
    5 ENQ    21 NAK    37 %    53 5    69 E    85 U   101 e   117 u 
    6 ACK    22 SYN    38 &    54 6    70 F    86 V   102 f   118 v 
    7 BEL    23 ETB    39 '    55 7    71 G    87 W   103 g   119 w 
    8 BS     24 CAN    40 (    56 8    72 H    88 X   104 h   120 x 
    9 HT     25 EM     41 )    57 9    73 I    89 Y   105 i   121 y 
   10 LF     26 SUB    42 *    58 :    74 J    90 Z   106 j   122 z 
   11 VT     27 ESC    43 +    59 ;    75 K    91 [   107 k   123 { 
   12 FF     28 FS     44 ,    60 <    76 L    92 \   108 l   124 | 
   13 CR     29 GS     45 -    61 =    77 M    93 ]   109 m   125 } 
   14 SO     30 RS     46 .    62 

初めの32文字は制御コードと呼ばれる通信用の文字。実際は目に見えない。

Pythonで実際にASCIIのコードポイントを確認してみる。

In [2]:
text = "ABC"
bytes = text.encode("ascii")
codepoints = list(bytes)
codepoints

[65, 66, 67]

A、B、Cそれぞれに65、66、67が割り当てられていることが分かる。なおコードポイントは16進数で表記することが多い。

In [3]:
codepoints_hex = list(map(hex, codepoints))
codepoints_hex

['0x41', '0x42', '0x43']

`0x`はそれが16進数であることを示す記号。16進法での41、42、43であることがわかる。

ASCIIは全部で$128=2^7$個の文字があり、1バイト=8ビットで1文字を表すことができる。1ビット余るが、そこは使わない。

### Unicode

世界中のあらゆる文字を収録した文字コード。

ASCIIではアルファベットを中心とした128文字しか扱えなかったが、世界には大量の言語とそれを記述する文字が存在する。我々が扱っている日本語も、常用漢字だけで約2000字と、大量の文字を持っている。

Unicodeに収録されている文字は日々増えており、2025年現在で約17万字の文字が収録されている。ちなみに最近追加された日本語の文字だと「㋿」がある。

Pythonでは`ord`関数でUnicodeのコードポイントを確認できる。

In [4]:
char = "あ"
codepoint = ord(char)
print(codepoint) # 10進法
print(hex(codepoint)) # 16進法

12354
0x3042


Unicodeのコードポイントは先頭に`U+`をつけて表記する。↑の「あ」だったら「U+3042」ってこと。

## 文字符号化方式

文字コードを元に実際に符号化を行う際の様々な手法について説明する。

ASCIIでは全てのコードポイントを1バイトで表すことで文字列を符号化した。一方Unicodeは文字の種類が多く、一意にコードポイントを表すためには複数のバイトが必要になる。このUnicodeもASCIIと同じように、全てのコードポイントを同じ長さの符号で表すことで簡単に符合化ができるが、このような固定長の符号による符号化は効率が悪い。可変長の符号を用いて一部のコードポイントを短い符号で表すことで、効率的な符号化が行える。

> このあたりの話は符号理論とも通ずる部分がある。例えばハフマン符号化では、出現頻度の高いパターンを分析した上で、そこに短い符号を割り当てることで効率よく符号化を行う。

### UTF-8

*Unicode Transformation Format 8-bit*

Unicodeの最も一般的な符号化手法。コードポイントを可変長の符号で表すことで効率的な符号化を実現する。

UTF-8ではUnicodeを次のように符号化する

| | 1バイト目 | 2バイト目 | 3バイト目 | 4バイト目 | 対応コードポイント範囲 |
|---|---|---|---|---|---|
| 1バイト文字 | 0xxxxxxx |  |  |  | U+0000 〜 U+007F |
| 2バイト文字 | 110xxxxx | 10xxxxxx |  |  | U+0080 〜 U+07FF |
| 3バイト文字 | 1110xxxx | 10xxxxxx | 10xxxxxx |  | U+0800 〜 U+FFFF |
| 4バイト文字 | 11110xxx | 10xxxxxx | 10xxxxxx | 10xxxxxx | U+10000 〜 U+10FFFF |

xにはコードポイントのビットが入る。

UTF-8では1バイト目の先頭ビット列で符号の長さを示す。1ビット目が0なら1バイト文字を表す。1ビット目が1なら複数バイト文字であることを示し、その後どれだけ1が続いたかで符号の長さが決まる。複数バイト文字の2バイト目以降は後続バイトと呼ばれ、全て10で始まる。

各バイトのprefix（先頭ビット列）をまとめると以下のようになる。

- `0`: 1バイト文字
- `10`: 後続バイト
- `110`: 2バイト文字
- `1110`: 3バイト文字
- `11110`: 4バイト文字

この符号化は一意復号可能性と自己同期性の両方を満たしている。どちらも符号理論における重要な性質である。一意復号可能性は文字通りで、符号化されたデータから元のデータを一意に復元できる性質である。自己同期性は、符号列を途中から読み始めても正しく復号できるという性質である。これは通信時のエラーなどでデータの一部が欠損した場合などにその後ろを正しく復元するために重要である。

例えばprefixでその文字のバイト数を明示しなかった場合、可変長の符号化においてはどこが文字の区切りかが分からなくなる為、一意復号可能性が失われる。またprefixで後続バイトであることを明示しない場合、途中から読んだときに文字の区切りが定まらなくなる可能性がある為、自己同期性が失われる。

1バイト文字はASCIIと同じ。利用できるコードポイント7ビット分にASCIIのコードポイントがそのまま入る。

In [5]:
text = "A"
print(list(map(hex, text.encode("ascii"))))
print(list(map(hex, text.encode("utf-8"))))

['0x41']
['0x41']


2バイト文字にはフランス語で使われるようなラテン拡張、キリル文字、ギリシャ文字などが固まっている。

In [6]:
char = "é"
list(map(hex, char.encode("utf-8")))

['0xc3', '0xa9']

3バイト文字には日本語、中国語、韓国語などが固まっている。

In [7]:
char = "あ"
list(map(hex, char.encode("utf-8")))

['0xe3', '0x81', '0x82']

4バイト文字には少し特殊な文字が固まっている。絵文字とか。

In [8]:
char = "😄"
list(map(hex, char.encode("utf-8")))

['0xf0', '0x9f', '0x98', '0x84']

実際に適当な文字をUTF-8で符号化してみよう。`.encode`は使わずに。

まず適当な文字を定義し、Unicodeのコードポイントを取得する。

In [9]:
char = "字"
codepoint = ord(char)
print(codepoint) # 10進法
print(hex(codepoint)) # 16進法
print(bin(codepoint)) # 2進法

23383
0x5b57
0b101101101010111


先の表を見ると、3バイト文字に該当することがわかる。ということはprefixは`1110`になる。

In [10]:
prefix = "1110"

で、3バイト文字はコードポイントが16ビット分あるので、16文字分0埋めする。

In [11]:
codepoint_bin_filled = str(bin(codepoint))[2:].zfill(16)
codepoint_bin_filled

'0101101101010111'

あとはこれを4,6,6ビットに分割して結合したらいい。

In [12]:
byte1 = prefix + codepoint_bin_filled[:4]
byte2 = "10" + codepoint_bin_filled[4:10]
byte3 = "10" + codepoint_bin_filled[10:]
bytes = [byte1, byte2, byte3]
bytes

['11100101', '10101101', '10010111']

3バイトの符号が得られた。確認してみよう。

In [13]:
bytes_true = list(map(lambda x: bin(x)[2:], char.encode("utf-8")))
print(bytes_true)
bytes == bytes_true

['11100101', '10101101', '10010111']


True

同じになったね。